In [41]:
import os
import json
from genson import SchemaBuilder
from sklearn.feature_extraction import DictVectorizer
from sklearn.cluster import AgglomerativeClustering
from pprint import pprint

import csv
import shutil
from pathlib import Path

In [ ]:


def get_matching_json_paths(top_level_directory):
    """
    Recursively finds all files matching the pattern '1i-*.json' 
    within the given top-level directory.
    
    :param top_level_directory: Path or string of the root directory to search.
    :return: A list of string paths matching the pattern.
    """
    # Convert string input to a Path object if necessary
    root_path = Path(top_level_directory)
    
    # Ensure the directory actually exists before searching
    if not root_path.exists() or not root_path.is_dir():
        print(f"Error: The directory '{top_level_directory}' does not exist or is not a directory.")
        return []

    # Find paths matching the pattern '1i-*.json' recursively
    # and convert them to standard string paths
    matching_paths = [str(file_path) for file_path in root_path.rglob("ai_*.json")]
    
    return matching_paths

# --- Example Usage ---
# directory_to_scan = "./proj-api-volume-dump"
# results = get_matching_json_paths(directory_to_scan)
# print(f"Found {len(results)} matching configuration files.")

In [39]:
ai_json_files = get_matching_json_paths(top_level_directory='./proj-api-volume-dump')

pprint(f'ai json file count: {len(ai_json_files)}')
pprint(ai_json_files)

'ai json file count: 1887'
['proj-api-volume-dump/7b261599-24c5-49fa-91f8-5c115a75974e/workspace/projects/Project_name_418987/ai_Project_name_418987.json',
 'proj-api-volume-dump/cbf09bd6-be92-4286-bfee-d848f4c07ac7/workspace/projects/Project_name_278105/ai_Project_name_278105.json',
 'proj-api-volume-dump/frb5azc94k4cs4zbk7eh2gfk10/workspace/projects/Dataset_Project_66524/ai_Dataset_Project_66524.json',
 'proj-api-volume-dump/b5pmdxc703wry5dww45mqdfk10/workspace/projects/project_name_183960/ai_project_name_183960.json',
 'proj-api-volume-dump/b5pmdxc703wry5dww45mqdfk10/workspace/projects/test/ai_test.json',
 'proj-api-volume-dump/e891f69d-c884-4a03-a0f5-0cb9a36483ae/workspace/projects/Project_name_370213/ai_Project_name_370213.json',
 'proj-api-volume-dump/e891f69d-c884-4a03-a0f5-0cb9a36483ae/workspace/projects/Project_name_111287/ai_Project_name_111287.json',
 'proj-api-volume-dump/e891f69d-c884-4a03-a0f5-0cb9a36483ae/workspace/projects/Project_name_72137/ai_Project_name_72137.json',

In [36]:
import csv
import json
import os
import re
from pathlib import Path


def flatten_json_to_paths(data, prefix=""):
    """Recursively flattens a JSON structure into a dictionary of dot-notated

    paths.
    E.g., {"models": [{"training": {"runtime": "python"}}]} becomes
    {"models.0.training.runtime": "python"}
    """
    flat_dict = {}

    if isinstance(data, dict):
        for key, value in data.items():
            new_prefix = f"{prefix}.{key}" if prefix else key
            flat_dict.update(flatten_json_to_paths(value, new_prefix))
    elif isinstance(data, list):
        for index, item in enumerate(data):
            new_prefix = f"{prefix}.{index}"
            flat_dict.update(flatten_json_to_paths(item, new_prefix))
    else:
        # Standardize empty or null values to text
        flat_dict[prefix] = str(data) if data is not None else ""

    return flat_dict


def clean_and_load(file_path):
    """Safely handles empty files, basic encoding fallbacks, and minor syntax

    repairs.
    """
    content = ""
    # Try reading with standard UTF-8 first
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read().strip()
    except UnicodeDecodeError:
        # Fallback to latin-1 to avoid decoding errors for unexpected encodings
        try:
            with open(file_path, "r", encoding="latin-1") as f:
                content = f.read().strip()
            # If it's a raw binary artifact with a false extension, drop it
            if content and not any(
                content.startswith(c) for c in ["{", "[", '"', "\n", "\r", " "]
            ):
                return None
        except Exception:
            return None

    if not content:
        return None

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        try:
            # Simple fallback to clean up trailing commas before closing braces/brackets
            cleaned = re.sub(r",\s*([\]}])", r"\1", content)
            return json.loads(cleaned)
        except Exception:
            return None


def convert_json_list_to_csv(json_paths_list, output_csv_path):
    """Takes a list of file paths, flattens their structural configurations,

    and exports them to a unified CSV grid.
    """
    all_flattened_files = []
    all_discovered_columns = set()

    print(f"Processing {len(json_paths_list)} files...")

    for path_str in json_paths_list:
        file_path = Path(path_str)
        if not file_path.exists():
            continue

        data = clean_and_load(file_path)
        if data is None:
            continue  # Skips unreadable, empty, or false binary artifacts

        # Flatten nested structure to single layer paths
        flat_data = flatten_json_to_paths(data)

        if flat_data:
            # Inject a tracking column for identification in your UI analysis tool
            flat_data["_source_file_path"] = str(file_path)
            all_flattened_files.append(flat_data)
            all_discovered_columns.update(flat_data.keys())

    if not all_flattened_files:
        print("No valid JSON data could be processed or extracted.")
        return

    # Ensure the identifier tracking column sits cleanly at the front of the CSV layout
    headers = ["_source_file_path"] + sorted(
        list(all_discovered_columns - {"_source_file_path"})
    )

    print(
        f"Writing structural matrix with {len(headers)} unique columns to: {output_csv_path}"
    )

    with open(output_csv_path, "w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=headers)
        writer.writeheader()

        for flat_entry in all_flattened_files:
            # Missing fields in specific schema versions automatically populate as empty strings
            writer.writerow(flat_entry)

    print("CSV generation successfully completed.")

In [40]:

# --- Example Pipeline Integration ---
if __name__ == "__main__":
    # Assuming you have the function from earlier or a pre-populated list
    # e.g., my_files = get_matching_json_paths("./proj-api-volume-dump")

    # For testing, you can pass an explicit list:
    # example_input_paths = [
    #     'proj-api-volume-dump/7b261599-24c5-49fa-91f8-5c115a75974e/workspace/projects/Project_name_418987/ai_Project_name_418987.json',
    #     'proj-api-volume-dump/cbf09bd6-be92-4286-bfee-d848f4c07ac7/workspace/projects/Project_name_278105/ai_Project_name_278105.json',
    # ]

    # output_destination = "./flattened_schema_matrix.csv"
    # convert_json_list_to_csv(example_input_paths, output_destination)

    convert_json_list_to_csv(ai_json_files, "./flattened_schema_matrix.csv")

Processing 1887 files...
Writing structural matrix with 1999 unique columns to: ./flattened_schema_matrix.csv
CSV generation successfully completed.


In [ ]:
def anonymize_and_map_json_files(json_paths_list, output_dir):
    """Reads a list of JSON file paths, copies and renames them to a flat index

    structure (ai_<index>.json), and generates a CSV file tracking the mapping
    between the original file path and the new indexed filename.

    :param json_paths_list: List of string paths pointing to your source JSON
    files.
    :param output_dir: Path or string specifying the target directory for
    outputs.
    """
    target_dir = Path(output_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

    csv_mapping_path = target_dir / "name_mapping.csv"
    mapping_rows = []

    print(
        f"Beginning segregation and renaming for {len(json_paths_list)} files..."
    )

    for index, original_path_str in enumerate(json_paths_list):
        source_file = Path(original_path_str)

        # Ensure the file exists before processing
        if not source_file.exists():
            print(f"Warning: File not found, skipping: {original_path_str}")
            continue

        # Define the target uniform filename schema
        new_filename = f"ai_{index}.json"
        destination_file = target_dir / new_filename

        try:
            # Copy the file to the flat output directory
            shutil.copy2(source_file, destination_file)

            # Record the exact spatial mapping for your downstream data mining
            mapping_rows.append(
                {
                    "index": index,
                    "anonymous_filename": new_filename,
                    "original_path": str(source_file),
                }
            )

        except IOError as e:
            print(
                f"Failed to process or copy file {original_path_str} due to system error: {e}"
            )

    # Write the tracking index grid to name_mapping.csv
    csv_headers = ["index", "anonymous_filename", "original_path"]
    try:
        with open(
            csv_mapping_path, "w", newline="", encoding="utf-8"
        ) as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames=csv_headers)
            writer.writeheader()
            writer.writerows(mapping_rows)

        print(
            f"Successfully processed files. Outputs and ledger mapped to: {target_dir}"
        )
    except IOError as e:
        print(f"Error writing the CSV mapping file: {e}")


# --- Quick Execution Check ---
if __name__ == "__main__":
    # Example input array derived from your search functions
    tracked_files = [
        "proj-api-volume-dump/c08a6da7-9afd-4c39-8ddf-12b85066ba8c/workspace/projects/ClonedAssetTracking/ai_get_started_asset_tracking_mlc_v10.json",
        "proj-api-volume-dump/ea66524a-2778-4966-a6ed-340abc5bc04b/workspace/projects/mlc_odr/ai_mlc_odr.json",
    ]

    # anonymize_and_map_json_files(
    #     tracked_files, "./processed_schema_workspace"
    # )

    anonymize_and_map_json_files(
        ai_json_files, "./processed_schema_workspace"
    )

    
    

Beginning segregation and renaming for 2 files...
Successfully processed files. Outputs and ledger mapped to: processed_schema_workspace
